# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



In [1]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


In [2]:
#python import
import sqlite3 as db
import pandas as pd
import contextlib

In [3]:
# Install the sqlite package for Ubuntu
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [4]:
# Download the Chinook sqlite database
%%bash
[ -f chinook.zip ] || curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -o chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [5]:
#Remove any old copy, then unzip
!rm -f chinook.db
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [6]:
# Check the files
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 12 14:29 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


### Run sql using jupyter magic

#### Count how many columns in each table

In [9]:
# List all the tables in the Chinook database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [11]:
# Show the columns in the albums table
%%script sqlite3 --column --header chinook.db
pragma table_info(albums);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    AlbumId   INTEGER        1                    1 
1    Title     NVARCHAR(160)  1                    0 
2    ArtistId  INTEGER        1                    0 


In [14]:
# How many columns are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_columns.
from pragma_table_info('albums')


Parse error near line 1: near ".": syntax error
  select count(*) as n_columns.        from pragma_table_info('albums')  
                error here ---^


CalledProcessError: Command 'b"select count(*) as n_columns.       \nfrom pragma_table_info('albums')  \n"' returned non-zero exit status 1.

In [15]:
# How many columns are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_columns from pragma_table_info('albums');
select 'artists'        as table_name, count(*) as n_columns from pragma_table_info('artists');
select 'customers'      as table_name, count(*) as n_columns from pragma_table_info('customers');
select 'employees'      as table_name, count(*) as n_columns from pragma_table_info('employees');
select 'genres'         as table_name, count(*) as n_columns from pragma_table_info('genres');
select 'invoice_items'  as table_name, count(*) as n_columns from pragma_table_info('invoice_items');
select 'invoices'       as table_name, count(*) as n_columns from pragma_table_info('invoices');
select 'media_types'    as table_name, count(*) as n_columns from pragma_table_info('media_types');
select 'playlist_track' as table_name, count(*) as n_columns from pragma_table_info('playlist_track');
select 'playlists'      as table_name, count(*) as n_columns from pragma_table_info('playlists');
select 'tracks'         as table_name, count(*) as n_columns from pragma_table_info('tracks');

table_name  n_columns
----------  ---------
albums      3        
table_name  n_columns
----------  ---------
artists     2        
table_name  n_columns
----------  ---------
customers   13       
table_name  n_columns
----------  ---------
employees   15       
table_name  n_columns
----------  ---------
genres      2        
table_name     n_columns
-------------  ---------
invoice_items  5        
table_name  n_columns
----------  ---------
invoices    9        
table_name   n_columns
-----------  ---------
media_types  2        
table_name      n_columns
--------------  ---------
playlist_track  2        
table_name  n_columns
----------  ---------
playlists   2        
table_name  n_columns
----------  ---------
tracks      9        


#### Count how many rows in each table

In [ ]:
# How many rows are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_rows
from albums;      # look at the table of rows

In [ ]:
# How many rows are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_rows from albums;
select 'artists'        as table_name, count(*) as n_rows from artists;
select 'customers'      as table_name, count(*) as n_rows from customers;
select 'employees'      as table_name, count(*) as n_rows from employees;
select 'genres'         as table_name, count(*) as n_rows from genres;
select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items;
select 'invoices'       as table_name, count(*) as n_rows from invoices;
select 'media_types'    as table_name, count(*) as n_rows from media_types;
select 'playlist_track' as table_name, count(*) as n_rows from playlist_track;
select 'playlists'      as table_name, count(*) as n_rows from playlists;
select 'tracks'         as table_name, count(*) as n_rows from tracks;

#### Break down columns

In [16]:
# Identifier columns are the Primary keys and Foreign keys in each table
# Which column is the Primary key in "tracks"?
%%script sqlite3 --column --header chinook.db
select name, pk
from pragma_table_info('tracks')
where pk > 0;


name     pk
-------  --
TrackId  1 


In [17]:
# Which columns in track tablet is the (Foreign key) point to other tables?
%%script sqlite3 --column --header chinook.db
select "from" as column_name, "table" as points_to
from pragma_foreign_key_list('tracks');

column_name  points_to  
-----------  -----------
MediaTypeId  media_types
GenreId      genres     
AlbumId      albums     


In [18]:
# Is TrackId unique? (the two numbers should match)
%%script sqlite3 --column --header chinook.db
select count(*)                as n_rows,
       count(distinct TrackId) as n_unique_ids
from tracks;

n_rows  n_unique_ids
------  ------------
3503    3503        


### Run SQl using DuckDB

In [19]:
import duckdb
import pandas as pd

con = duckdb.connect()                                  # an empty DuckDB session in memory
con.sql("INSTALL sqlite; LOAD sqlite;")                 # add-on that lets DuckDB read SQLite files
con.sql("ATTACH 'chinook.db' AS chinook (TYPE sqlite)") # plug in your file and call it 'chinook'
con.sql("USE chinook")                                  # make it the default, so you can write 'albums'

## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods.  The initial purpose is to get familiar with SQL.




### Basic Queries


#### SELECT (with * and with column names)


#### WHERE


#### AND


#### OR


#### LIKE (with % or _ wildcard)


#### BETWEEN


#### LIMIT



### Sorting and Grouping


#### ORDER BY


#### DISTINCT


#### GROUP BY



### Aggregates


#### MAX


#### MIN


#### SUM


#### AVG


#### COUNT



## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



## EXTRA CREDIT:
* Use a CTE
* Use a query that joins two or more tables.
* Make a model to see if you can predict something ( e.g. GNB classifier, linear regression )
* Come up with something else cool to do with your data
